# Fiscal Calendar 충돌 진단 노트북

**목적**: 2,000 ticker 중 DCFModel 의 calendar Q 정규화 시 actual + forecast 가 충돌하는 ticker 를 식별

**문제 정의**:
- DCFModel v10.5 의 `_dedup_to_calendar_q` 가 모든 dates 를 calendar Q end (3/31, 6/30, 9/30, 12/31) 로 변환
- NVDA 같은 13W fiscal calendar ticker 의 경우:
  - actual 마지막 (2026-01-25, fiscal Q4 FY26) → calendar 2026Q1
  - forecast 첫 (2026-03-31, calendar Q1 end) → calendar 2026Q1
  - **같은 calendar Q 충돌** → Excel 시트에 분기 중복 표시

**조사 항목**:
1. 충돌 발생 ticker 수와 비중
2. Fiscal year end 월별 분포
3. 13W ticker 비율
4. 충돌 패턴별 분류 (Phase 1 옵션 1 vs Phase 2 결정용)

**결과 활용**:
- 충돌 ticker < 200 → Phase 1 (옵션 1, actual 우선 dedup) 충분
- 충돌 ticker > 500 → Phase 2 (전면 재구성, fiscal Q 보존) 필수
- 200-500 → 케이스 분석 후 결정

## Cell 1 · 환경 설정 & DB 연결

In [9]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",
]

def _setup_path() -> str:
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()

    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root 자동 감지 : {root}")
            return root

    for candidate in _CANDIDATE_ROOTS:
        if os.path.isdir(candidate) and os.path.isdir(os.path.join(candidate, "DATA")):
            if candidate not in sys.path:
                sys.path.insert(0, candidate)
            print(f"[PATH] root 후보 경로 : {candidate}")
            return candidate

    raise EnvironmentError("DATA 폴더를 찾을 수 없습니다.")

_ROOT = _setup_path()
print(f"[확인] 프로젝트 루트  : {_ROOT}")

[PATH] root 자동 감지 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] 프로젝트 루트  : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy


In [10]:
import pandas as pd
import numpy as np
from datetime import datetime
import pymysql

from DATA.config import get_db_info

db_info = get_db_info()
print(f"[DB] {db_info.get('host')}:{db_info.get('port')} / {db_info.get('database')}")

def get_conn():
    return pymysql.connect(
        host        = db_info["host"],
        port        = int(db_info["port"]),
        user        = db_info["user"],
        password    = db_info["password"],
        database    = db_info["database"],
        charset     = "utf8mb4",
        cursorclass = pymysql.cursors.DictCursor,
    )

def run_query(sql: str, params: tuple = None) -> pd.DataFrame:
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(sql, params or ())
            rows = cur.fetchall()
        return pd.DataFrame(rows) if rows else pd.DataFrame()
    finally:
        conn.close()

# 연결 테스트
df_test = run_query("SELECT 1 AS ok")
print(f"[OK] DB 연결: {df_test.iloc[0, 0]}")

TABLE = "us_revenue_forecast_data"

[DB] 192.168.0.230:3307 / investar
[OK] DB 연결: 1


## Cell 2 · ★ 핵심 진단: actual vs forecast 의 calendar Q 충돌

각 ticker 에 대해:
- actual 의 마지막 분기 → calendar Q
- 가장 최근 forecast 의 첫 분기 → calendar Q
- 둘이 같은 calendar Q 이면 **CONFLICT**

In [11]:
sql = """
    WITH actual_last AS (
        SELECT ticker, MAX(date) AS last_date
        FROM us_revenue_forecast_data
        WHERE data_type = 'actual'
        GROUP BY ticker
    ),
    forecast_first AS (
        SELECT 
            fc.ticker,
            MIN(fc.date) AS first_date,
            fc.forecast_date
        FROM us_revenue_forecast_data fc
        INNER JOIN (
            SELECT ticker, MAX(forecast_date) AS max_fd
            FROM us_revenue_forecast_data
            WHERE data_type = 'forecast'
            GROUP BY ticker
        ) latest ON fc.ticker = latest.ticker AND fc.forecast_date = latest.max_fd
        WHERE fc.data_type = 'forecast'
        GROUP BY fc.ticker, fc.forecast_date
    )
    SELECT 
        a.ticker,
        a.last_date AS actual_last,
        f.first_date AS forecast_first,
        f.forecast_date,
        DATEDIFF(f.first_date, a.last_date) AS gap_days,
        YEAR(a.last_date) AS act_y, QUARTER(a.last_date) AS act_q,
        YEAR(f.first_date) AS fc_y, QUARTER(f.first_date) AS fc_q,
        CASE 
            WHEN YEAR(a.last_date)=YEAR(f.first_date) AND QUARTER(a.last_date)=QUARTER(f.first_date)
            THEN 'CONFLICT'
            ELSE 'OK'
        END AS status
    FROM actual_last a
    INNER JOIN forecast_first f ON a.ticker = f.ticker
    ORDER BY status DESC, gap_days ASC, a.ticker
"""

df_all = run_query(sql)
print(f"[전체] 분석 ticker: {len(df_all)}개")

# 상태별 통계
print()
status_counts = df_all["status"].value_counts()
print("[상태별 분포]")
for status, n in status_counts.items():
    pct = n / len(df_all) * 100
    print(f"  {status:<10s}: {n:>5d}  ({pct:5.1f}%)")

# ★ 핵심 결과
n_conflict = len(df_all[df_all["status"] == "CONFLICT"])
n_total = len(df_all)
print()
print("=" * 60)
print(f"★ 충돌 ticker: {n_conflict}개  ({n_conflict/n_total*100:.1f}%)")
print("=" * 60)

# 권장 액션
print()
if n_conflict < 200:
    print(f"  → Phase 1 (옵션 1) 권장 — actual 우선 dedup 으로 해결 가능")
    print(f"  → 영향 ticker 가 적어 임시 fix 로 충분")
elif n_conflict > 500:
    print(f"  → Phase 2 (전면 재구성) 필수 — fiscal Q 보존 방식으로 변경")
    print(f"  → 영향 ticker 가 많아 임시 fix 는 불충분")
else:
    print(f"  → Phase 1 또는 Phase 2 — 충돌 패턴 분석 후 결정")
    print(f"  → 다음 셀들의 결과 검토 필요")

[전체] 분석 ticker: 1944개

[상태별 분포]
  OK        :  1603  ( 82.5%)
  CONFLICT  :   341  ( 17.5%)

★ 충돌 ticker: 341개  (17.5%)

  → Phase 1 또는 Phase 2 — 충돌 패턴 분석 후 결정
  → 다음 셀들의 결과 검토 필요


## Cell 3 · 충돌 ticker 의 fiscal pattern 분류

충돌 ticker 들이 어떤 fiscal calendar 를 가지고 있는지 분석.

In [12]:
df_conflict = df_all[df_all["status"] == "CONFLICT"].copy()

if df_conflict.empty:
    print("[충돌 없음] 모든 ticker 정상")
else:
    df_conflict["actual_last"] = pd.to_datetime(df_conflict["actual_last"])
    df_conflict["forecast_first"] = pd.to_datetime(df_conflict["forecast_first"])
    
    # actual 마지막의 dow (day of week)
    df_conflict["act_dow"] = df_conflict["actual_last"].dt.day_name()
    df_conflict["act_month"] = df_conflict["actual_last"].dt.month
    df_conflict["act_day"] = df_conflict["actual_last"].dt.day
    df_conflict["act_mmdd"] = df_conflict["actual_last"].dt.strftime("%m-%d")
    
    # Fiscal pattern 분류
    def classify_pattern(row):
        mmdd = row["act_mmdd"]
        dow = row["act_dow"]
        
        # 1. Calendar Q end (3/31, 6/30, 9/30, 12/31) — 12월 fiscal year + 일반
        if mmdd in ["03-31", "06-30", "09-30", "12-31"]:
            return "calendar_q_end"
        
        # 2. Month end (다른 월의 마지막 날) — regular month-end fiscal
        # 1/31, 4/30, 7/31, 10/31 등
        if row["act_day"] >= 28:
            return "month_end_fiscal"
        
        # 3. 13W fiscal calendar — 토요일/일요일 mid-month
        if dow in ["Saturday", "Sunday"]:
            return "13W_fiscal_weekend"
        
        # 4. 13W fiscal — 다른 요일 (드뭄)
        return "13W_fiscal_weekday"
    
    df_conflict["pattern"] = df_conflict.apply(classify_pattern, axis=1)
    
    print(f"[충돌 ticker {len(df_conflict)}개의 fiscal pattern 분류]")
    pattern_counts = df_conflict["pattern"].value_counts()
    for p, n in pattern_counts.items():
        pct = n / len(df_conflict) * 100
        print(f"  {p:<25s}: {n:>5d}  ({pct:5.1f}%)")
    
    print()
    print("[패턴별 sample (각 5개)]")
    for pattern in pattern_counts.index:
        sub = df_conflict[df_conflict["pattern"] == pattern].head(5)
        print(f"\n  [{pattern}]")
        cols = ["ticker", "actual_last", "forecast_first", "gap_days", "act_dow"]
        sub_show = sub[cols].copy()
        sub_show["actual_last"] = sub_show["actual_last"].dt.date
        sub_show["forecast_first"] = sub_show["forecast_first"].dt.date
        print(sub_show.to_string(index=False))

[충돌 ticker 341개의 fiscal pattern 분류]
  month_end_fiscal         :   201  ( 58.9%)
  13W_fiscal_weekend       :   100  ( 29.3%)
  13W_fiscal_weekday       :    40  ( 11.7%)

[패턴별 sample (각 5개)]

  [month_end_fiscal]
ticker actual_last forecast_first  gap_days  act_dow
  BJRI  2025-12-30     2025-12-31         1  Tuesday
  CAKE  2025-12-30     2025-12-31         1  Tuesday
  PBPB  2025-06-29     2025-06-30         1   Sunday
  PGTI  2023-12-30     2023-12-31         1 Saturday
  TXRH  2025-12-30     2025-12-31         1  Tuesday

  [13W_fiscal_weekend]
ticker actual_last forecast_first  gap_days  act_dow
   HBI  2025-09-27     2025-09-30         3 Saturday
     K  2025-09-27     2025-09-30         3 Saturday
  KOPN  2025-09-27     2025-09-30         3 Saturday
   ODP  2025-09-27     2025-09-30         3 Saturday
   AMD  2025-12-27     2025-12-31         4 Saturday

  [13W_fiscal_weekday]
ticker actual_last forecast_first  gap_days  act_dow
  JBSS  2025-12-25     2025-12-28         3 Thurs

## Cell 4 · 정상 ticker 의 fiscal pattern 분포

비교용: 충돌 안 일어난 ticker 들도 fiscal pattern 별 분포 확인.

In [13]:
df_ok = df_all[df_all["status"] == "OK"].copy()
df_ok["actual_last"] = pd.to_datetime(df_ok["actual_last"])
df_ok["forecast_first"] = pd.to_datetime(df_ok["forecast_first"])
df_ok["act_dow"] = df_ok["actual_last"].dt.day_name()
df_ok["act_day"] = df_ok["actual_last"].dt.day
df_ok["act_mmdd"] = df_ok["actual_last"].dt.strftime("%m-%d")

def classify_pattern(row):
    mmdd = row["act_mmdd"]
    dow = row["act_dow"]
    if mmdd in ["03-31", "06-30", "09-30", "12-31"]:
        return "calendar_q_end"
    if row["act_day"] >= 28:
        return "month_end_fiscal"
    if dow in ["Saturday", "Sunday"]:
        return "13W_fiscal_weekend"
    return "13W_fiscal_weekday"

df_ok["pattern"] = df_ok.apply(classify_pattern, axis=1)

print(f"[정상 ticker {len(df_ok)}개의 fiscal pattern 분포]")
pattern_counts_ok = df_ok["pattern"].value_counts()
for p, n in pattern_counts_ok.items():
    pct = n / len(df_ok) * 100
    print(f"  {p:<25s}: {n:>5d}  ({pct:5.1f}%)")

# 충돌 vs 정상 비교
print()
print("=" * 60)
print("패턴별 충돌 발생률")
print("=" * 60)
print(f"  {'pattern':<25s} {'conflict':>8s} {'ok':>8s} {'total':>8s} {'conflict %':>10s}")
print(f"  {'-'*25} {'-'*8} {'-'*8} {'-'*8} {'-'*10}")

if not df_conflict.empty:
    pat_c = df_conflict["pattern"].value_counts()
else:
    pat_c = pd.Series([], dtype=int)

pat_o = df_ok["pattern"].value_counts()
all_patterns = sorted(set(list(pat_c.index) + list(pat_o.index)))

for p in all_patterns:
    n_c = pat_c.get(p, 0)
    n_o = pat_o.get(p, 0)
    n_t = n_c + n_o
    rate = n_c / n_t * 100 if n_t > 0 else 0
    print(f"  {p:<25s} {n_c:>8d} {n_o:>8d} {n_t:>8d} {rate:>9.1f}%")

[정상 ticker 1603개의 fiscal pattern 분포]
  calendar_q_end           :  1583  ( 98.8%)
  month_end_fiscal         :    11  (  0.7%)
  13W_fiscal_weekend       :     5  (  0.3%)
  13W_fiscal_weekday       :     4  (  0.2%)

패턴별 충돌 발생률
  pattern                   conflict       ok    total conflict %
  ------------------------- -------- -------- -------- ----------
  13W_fiscal_weekday              40        4       44      90.9%
  13W_fiscal_weekend             100        5      105      95.2%
  calendar_q_end                   0     1583     1583       0.0%
  month_end_fiscal               201       11      212      94.8%


## Cell 5 · Gap 분포 분석

actual 마지막 ↔ forecast 첫 사이의 일수 분포. gap 이 음수 또는 매우 작으면 충돌.

In [14]:
df_all["gap_days"] = df_all["gap_days"].astype(int)

# Gap 분포 buckets
def gap_bucket(gap):
    if gap <= 0:
        return "1. 음수/0 (forecast 가 actual 과 같거나 옛날)"
    elif gap < 30:
        return "2. 1-29일 (충돌 위험)"
    elif gap < 60:
        return "3. 30-59일"
    elif gap < 80:
        return "4. 60-79일 (충돌 위험)"
    elif 80 <= gap <= 100:
        return "5. 80-100일 (정상)"
    elif gap < 120:
        return "6. 101-119일"
    else:
        return "7. 120일 이상 (actual 누락)"

df_all["gap_bucket"] = df_all["gap_days"].apply(gap_bucket)

print("[Gap 분포]")
bucket_counts = df_all["gap_bucket"].value_counts().sort_index()
for bucket, n in bucket_counts.items():
    pct = n / len(df_all) * 100
    bar = "█" * int(pct / 2)
    print(f"  {bucket:<50s} {n:>5d} ({pct:5.1f}%) {bar}")

# Gap 이 음수인 ticker (forecast 가 actual 보다 옛날 — 매우 비정상)
n_negative = (df_all["gap_days"] <= 0).sum()
if n_negative > 0:
    print()
    print(f"[★ 주의] gap_days <= 0 인 ticker: {n_negative}개")
    print("   → forecast 가 actual 마지막보다 같거나 옛날 분기로 시작")
    print("   → forecast 갱신 권장")
    sample = df_all[df_all["gap_days"] <= 0].head(10)
    print()
    print("   Sample:")
    cols = ["ticker", "actual_last", "forecast_first", "gap_days"]
    print(sample[cols].to_string(index=False))

[Gap 분포]
  2. 1-29일 (충돌 위험)                                     135 (  6.9%) ███
  3. 30-59일                                            163 (  8.4%) ████
  4. 60-79일 (충돌 위험)                                     24 (  1.2%) 
  5. 80-100일 (정상)                                     1609 ( 82.8%) █████████████████████████████████████████
  7. 120일 이상 (actual 누락)                                13 (  0.7%) 


## Cell 6 · 종합 진단 — Phase 결정 가이드

위 셀들의 결과를 종합하여 다음 단계 결정.

In [15]:
print("=" * 70)
print("  종합 진단 — Phase 결정 가이드")
print("=" * 70)

n_total = len(df_all)
n_conflict = len(df_all[df_all["status"] == "CONFLICT"])
n_ok = n_total - n_conflict
n_negative_gap = (df_all["gap_days"] <= 0).sum()

print(f"\n  · 분석 ticker            : {n_total:,}개")
print(f"  · 충돌 (CONFLICT)        : {n_conflict:,}개  ({n_conflict/n_total*100:.1f}%)")
print(f"  · 정상 (OK)              : {n_ok:,}개  ({n_ok/n_total*100:.1f}%)")
print(f"  · gap ≤ 0 (forecast 갱신 필요): {n_negative_gap:,}개")

# 패턴별 충돌
if not df_conflict.empty:
    n_13w = len(df_conflict[df_conflict["pattern"].isin(["13W_fiscal_weekend", "13W_fiscal_weekday"])])
    n_month_end = len(df_conflict[df_conflict["pattern"] == "month_end_fiscal"])
    n_calendar = len(df_conflict[df_conflict["pattern"] == "calendar_q_end"])
else:
    n_13w = n_month_end = n_calendar = 0

print(f"\n  [충돌 패턴]")
print(f"    13W fiscal calendar     : {n_13w}개")
print(f"    Month-end fiscal        : {n_month_end}개")
print(f"    Calendar Q end (의외)   : {n_calendar}개")

print()
print("=" * 70)
print("  ★ Phase 결정")
print("=" * 70)

if n_conflict < 200:
    decision = "Phase 1 권장"
    reason = ("영향 ticker 가 < 200 으로 적어 임시 fix 로 충분.\n"
              "  · DCFModel 의 _dedup_to_calendar_q 에 'actual 우선 dedup' 추가\n"
              "  · forecast 1 분기 손실 (8Q → 7Q) 발생 가능\n"
              "  · 작업 시간: 30분")
elif n_conflict > 500:
    decision = "Phase 2 필수"
    reason = ("영향 ticker 가 > 500 으로 많아 임시 fix 는 불충분.\n"
              "  · DCFModel 전체를 fiscal Q 기반으로 재구성\n"
              "  · 학술 표준 (Bloomberg, Compustat 방식) 적용\n"
              "  · 작업 시간: 4-6시간 + 검증")
else:
    decision = "Phase 1 또는 Phase 2"
    reason = ("영향 ticker 가 200-500 으로 중간 — 패턴 분석 후 결정.\n"
              "  · 만약 13W ticker 가 많으면 Phase 2 권장 (체계적 해결)\n"
              "  · 만약 다양한 패턴이면 Phase 1 도 고려 가능")

print(f"\n  결정: {decision}")
print(f"\n  근거:")
print(f"  {reason}")

print()
print("=" * 70)
print("  추가 권고")
print("=" * 70)

if n_negative_gap > 0:
    print(f"\n  · gap ≤ 0 인 {n_negative_gap}개 ticker 는 forecast 갱신 권장")
    print(f"    → forecast 노트북 v4 의 Cell 13 에서 RUN_TICKERS 로 지정 후 재실행")
    
print()
if n_conflict > 0:
    pct_total = n_conflict / n_total * 100
    print(f"  · 현재 batch valuation 결과 중 {pct_total:.1f}% 가 분기 중복 영향 받을 수 있음")
    print(f"    → 호영님이 최종 결과를 사용하시기 전에 fix 적용 권장")

  종합 진단 — Phase 결정 가이드

  · 분석 ticker            : 1,944개
  · 충돌 (CONFLICT)        : 341개  (17.5%)
  · 정상 (OK)              : 1,603개  (82.5%)
  · gap ≤ 0 (forecast 갱신 필요): 0개

  [충돌 패턴]
    13W fiscal calendar     : 140개
    Month-end fiscal        : 201개
    Calendar Q end (의외)   : 0개

  ★ Phase 결정

  결정: Phase 1 또는 Phase 2

  근거:
  영향 ticker 가 200-500 으로 중간 — 패턴 분석 후 결정.
  · 만약 13W ticker 가 많으면 Phase 2 권장 (체계적 해결)
  · 만약 다양한 패턴이면 Phase 1 도 고려 가능

  추가 권고

  · 현재 batch valuation 결과 중 17.5% 가 분기 중복 영향 받을 수 있음
    → 호영님이 최종 결과를 사용하시기 전에 fix 적용 권장


## Cell 7 · 충돌 ticker 전체 리스트 (옵션, 디테일 분석용)

충돌 ticker 의 전체 리스트를 CSV 로 export 해두면 추후 패치 검증에 유용.

In [8]:
if not df_conflict.empty:
    # 보기 쉽게 정리
    df_export = df_conflict.copy()
    df_export["actual_last"] = df_export["actual_last"].dt.date
    df_export["forecast_first"] = df_export["forecast_first"].dt.date
    df_export["forecast_date"] = pd.to_datetime(df_export["forecast_date"]).dt.date
    
    cols = ["ticker", "actual_last", "forecast_first", "forecast_date",
            "gap_days", "act_dow", "act_mmdd", "pattern"]
    df_export = df_export[cols]
    
    print(f"[충돌 ticker 전체 리스트 ({len(df_export)}개)]")
    print()
    print(df_export.head(30).to_string(index=False))
    
    if len(df_export) > 30:
        print(f"\n   ... ({len(df_export) - 30}개 더 있음)")
    
    # CSV export (선택)
    # csv_path = r"C:\reports\fiscal_calendar_conflict_tickers.csv"
    # df_export.to_csv(csv_path, index=False, encoding="utf-8-sig")
    # print(f"\n[CSV 저장] {csv_path}")
else:
    print("[충돌 없음] 모든 ticker 정상 — fix 불필요")

[충돌 ticker 전체 리스트 (352개)]

ticker actual_last forecast_first forecast_date  gap_days  act_dow act_mmdd            pattern
   AZO  2026-02-14     2026-02-15    2026-04-30         1 Saturday    02-14 13W_fiscal_weekend
  BJRI  2025-12-30     2025-12-31    2026-04-04         1  Tuesday    12-30   month_end_fiscal
  CAKE  2025-12-30     2025-12-31    2026-04-03         1  Tuesday    12-30   month_end_fiscal
  CENT  2025-12-27     2025-12-28    2026-04-03         1 Saturday    12-27 13W_fiscal_weekend
  COST  2026-02-15     2026-02-16    2026-04-30         1   Sunday    02-15 13W_fiscal_weekend
  PBPB  2025-06-29     2025-06-30    2026-04-04         1   Sunday    06-29   month_end_fiscal
  PGTI  2023-12-30     2023-12-31    2026-04-03         1 Saturday    12-30   month_end_fiscal
  TXRH  2025-12-30     2025-12-31    2026-04-03         1  Tuesday    12-30   month_end_fiscal
   WLY  2026-01-31     2026-02-01    2026-04-03         1 Saturday    01-31   month_end_fiscal
  WLYB  2026-01-31     